# The Bedrock Flows lifecycle

A **flow** is a workflow you *configure* instead of code: nodes (input, prompt, output, and more) joined by connections. This notebook builds the smallest useful flow, input → prompt → output, and calls each Flows API by hand so you can see the whole lifecycle:

`create_flow` → `prepare_flow` → `create_flow_version` → `create_flow_alias` → `invoke_flow` → delete

> **Teaching/Learning Tip:** the console builder and the API are two views of the same thing. Run this notebook but skip the cleanup cell at the end, then open the flow in the Bedrock console (Flows). The canvas shows exactly the nodes and connections defined in code. That is the "no code, swap models in a dropdown" moment: click the prompt node and you can change the model, the template, and the inference settings without touching Python.

## Setup: clients and an execution role

A flow runs under an IAM **execution role** that Bedrock assumes. `flow_utils.ensure_role` creates `BedrockFlowsSampleRole`, trusted only by Bedrock flows in your account and region, with runtime permissions only (invoke models, prompts, knowledge bases, agents, guardrails). The account and region come from your login, nothing is hard-coded.

In [ ]:
import sys
sys.path.append("..")  # flow_utils.py lives one folder up
import flow_utils

session, region, account = flow_utils.get_session_info()  # from your current AWS login
agent = session.client("bedrock-agent")            # control plane: create and manage flows
runtime = session.client("bedrock-agent-runtime")  # data plane: invoke flows
iam = session.client("iam")
model_id = flow_utils.default_model_id(region, account)
print(f"Account {account} | region {region} | model {model_id}")

role_arn = flow_utils.ensure_role(iam, region, account)

## The flow definition: nodes and connections

Three nodes and two connections:

- **FlowInputNode** (`Input`): entry point, receives a string
- **Explain** (`Prompt`): an inline template, `Explain {{topic}} in two sentences.`, run by a model
- **FlowOutputNode** (`Output`): returns the model's text

A connection links one node's *output* to another node's *input*. The prompt's `topic` input reads the whole input document with the expression `$.data`.

> **Teaching/Learning Tip:** node and connection names are just labels you choose, but the `sourceOutput`/`targetInput` names in a connection must match the output and input names declared on the nodes, and the types must line up. Most "flow won't prepare" errors are a mismatch here.

In [ ]:
definition = {
    "nodes": [
        {"name": "FlowInputNode", "type": "Input",
         "outputs": [{"name": "document", "type": "String"}]},
        {"name": "Explain", "type": "Prompt",
         "inputs": [{"name": "topic", "type": "String", "expression": "$.data"}],
         "outputs": [{"name": "modelCompletion", "type": "String"}],
         "configuration": {"prompt": {"sourceConfiguration": {"inline": {
             "modelId": model_id,
             "templateType": "TEXT",
             "inferenceConfiguration": {"text": {"temperature": 0.7}},
             "templateConfiguration": {"text": {"text": "Explain {{topic}} in two sentences."}},
         }}}}},
        {"name": "FlowOutputNode", "type": "Output",
         "inputs": [{"name": "document", "type": "String", "expression": "$.data"}]},
    ],
    "connections": [
        {"name": "InputToExplain", "source": "FlowInputNode", "target": "Explain", "type": "Data",
         "configuration": {"data": {"sourceOutput": "document", "targetInput": "topic"}}},
        {"name": "ExplainToOutput", "source": "Explain", "target": "FlowOutputNode", "type": "Data",
         "configuration": {"data": {"sourceOutput": "modelCompletion", "targetInput": "document"}}},
    ],
}

## 1. Create the flow

`create_flow` registers the definition as a **draft**. A newly created IAM role can take a few seconds to become usable, so the cell retries on that specific error.

In [ ]:
import time

flow_id = None
for attempt in range(6):
    try:
        flow_id = agent.create_flow(
            name="LifecycleDemoFlow",
            description="Smallest useful flow: input -> prompt -> output",
            executionRoleArn=role_arn,
            definition=definition,
        )["id"]
        break
    except agent.exceptions.ValidationException as e:
        if "role" not in str(e).lower() or attempt == 5:
            raise
        time.sleep(5)
print("flow_id:", flow_id)

## 2. Prepare the flow

`prepare_flow` compiles and validates the draft. It is **asynchronous**: it returns immediately and the flow moves `Preparing` → `Prepared` (or `Failed`). Poll until it is done before creating a version.

> **Teaching/Learning Tip:** if you ever edit a flow in the console, click **Save** then **Prepare** before you test. Otherwise you are testing a stale version.

In [ ]:
agent.prepare_flow(flowIdentifier=flow_id)
while True:
    status = agent.get_flow(flowIdentifier=flow_id)["status"]
    print("status:", status)
    if status == "Prepared":
        break
    if status == "Failed":
        raise RuntimeError("Flow failed to prepare")
    time.sleep(3)

## 3. Version and alias

A **version** is an immutable snapshot of the prepared draft. An **alias** is a stable name that points at a version. Applications call the alias.

> **Teaching/Learning Tip:** an alias works like a Lambda alias. Publish a new version, repoint the alias, and callers pick it up without any code change. Note the console **Test flow** pane runs the *draft*, while an API call through the alias runs the *published version*. Console edits do not change what the alias serves until you publish a new version and update the alias.

In [ ]:
version = agent.create_flow_version(flowIdentifier=flow_id)["version"]
alias_id = agent.create_flow_alias(
    flowIdentifier=flow_id,
    name="live",
    routingConfiguration=[{"flowVersion": version}],
)["id"]
print("version:", version, "| alias_id:", alias_id)

## 4. Invoke the flow

`invoke_flow` streams events back. The input must match the **Input node's declared type**: a plain string here. Watch for `flowOutputEvent` (the result) and `flowCompletionEvent` (the status).

In [ ]:
response = runtime.invoke_flow(
    flowIdentifier=flow_id,
    flowAliasIdentifier=alias_id,
    inputs=[{
        "content": {"document": "Amazon Bedrock Flows"},
        "nodeName": "FlowInputNode",
        "nodeOutputName": "document",
    }],
)
for event in response["responseStream"]:
    if "flowOutputEvent" in event:
        print("OUTPUT:", event["flowOutputEvent"]["content"]["document"])
    elif "flowCompletionEvent" in event:
        print("completion:", event["flowCompletionEvent"]["completionReason"])

> **Teaching/Learning Tip:** the same call from a terminal proves this is a real API and not just a console feature:
>
> ```bash
> aws bedrock-agent-runtime invoke-flow \
>   --flow-identifier <flow_id> --flow-alias-identifier <alias_id> \
>   --inputs '[{"nodeName":"FlowInputNode","nodeOutputName":"document","content":{"document":"Amazon Bedrock Flows"}}]' \
>   /dev/stdout
> ```

## Cleanup

Delete the alias, then the version, then the flow. An alias or version blocks deleting what it points to.
The sample IAM role is left in place so the next module can reuse it. Delete it with
`flow_utils.delete_role(iam)` when you are completely done.

In [ ]:
flow_utils.cleanup_flow(agent, flow_id, alias_id, version)